# Практическая работа 1

Выполнены два задания:

1. Собственный парсинг документов (PDF, DOCX, TXT) без библиотеки `unstructured`.
2. Сравнение векторных баз данных Qdrant и ChromaDB.

Все данные создаются в самом ноутбуке, поэтому его можно запустить с нуля без ключей и доступа к внешним сервисам.

In [1]:
# pip install pypdf python-docx reportlab chromadb qdrant-client psutil pandas numpy

## Задание 1. Собственный парсинг документов

### 1.1. Тестовые файлы
Создаются 3 PDF (обычный текст, таблица, список с переносами слов), 2 DOCX (текст со списком, таблица) и один TXT.

In [2]:
import os, glob, textwrap
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import SimpleDocTemplate, Table, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle
from docx import Document

os.makedirs("data", exist_ok=True)

font_path = (glob.glob("/usr/share/fonts/**/DejaVuSans.ttf", recursive=True) + glob.glob("C:/Windows/Fonts/arial.ttf"))[0]
pdfmetrics.registerFont(TTFont("RU", font_path))

text = ("Канцелярские товары нужны каждому школьнику и офисному сотруднику. "
        "Тетрадь в клетку подходит для математики, а тетрадь в линейку для русского языка. "
        "При выборе ручки обращайте внимание на цвет чернил и удобство корпуса. ") * 12

# PDF 1: текст на двух страницах
c = canvas.Canvas("data/text.pdf", pagesize=A4)
c.setFont("RU", 11)
for page in range(2):
    y = 800
    for line in textwrap.wrap(text, 80):
        c.drawString(40, y, line)
        y -= 16
        if y < 40:
            break
    c.showPage()
c.save()

# PDF 2: таблица
rows = [["Товар", "Цена", "Остаток"]] + [[f"Товар {i}", f"{100 + i * 10} руб.", str(i * 3)] for i in range(1, 21)]
SimpleDocTemplate("data/table.pdf", pagesize=A4).build(
    [Paragraph("Прайс-лист", ParagraphStyle("s", fontName="RU")), Spacer(1, 12),
     Table(rows, style=[("FONTNAME", (0, 0), (-1, -1), "RU")])])

# PDF 3: список и слова с переносом через дефис
c = canvas.Canvas("data/list.pdf", pagesize=A4)
c.setFont("RU", 11)
y = 800
for line in ["Как выбрать тетрадь:", "- смотрите на плотность бумаги", "- проверьте раз-", "метку клетки",
             "- выбирайте нужное количест-", "во листов", "1. Для математики: клетка", "2. Для русского: линейка"]:
    c.drawString(40, y, line)
    y -= 18
c.save()

# DOCX 1: заголовок, абзац, маркированный список
d = Document()
d.add_heading("Инструкция по канцтоварам", 1)
d.add_paragraph(text[:600])
d.add_paragraph("Первый пункт", style="List Bullet")
d.add_paragraph("Второй пункт", style="List Bullet")
d.save("data/guide.docx")

# DOCX 2: таблица
d = Document()
d.add_paragraph("Каталог")
t = d.add_table(rows=4, cols=3)
for i, r in enumerate([["Название", "Цена", "Цвет"], ["Ручка", "50", "синий"], ["Тетрадь", "30", "белый"], ["Маркер", "70", "красный"]]):
    for j, v in enumerate(r):
        t.cell(i, j).text = v
d.add_paragraph("Конец каталога.   Лишние   пробелы   здесь.")
d.save("data/catalog.docx")

# TXT
with open("data/note.txt", "w", encoding="utf-8", newline="") as f:
    f.write("Заметка.\r\n\r\n\r\nТекст   с   лишними пробелами.\r\nВторая строка.")

print(sorted(os.listdir("data")))

['catalog.docx', 'guide.docx', 'list.pdf', 'note.txt', 'table.pdf', 'text.pdf']


### 1.2. Загрузка файлов
Для каждого формата написана отдельная функция. Функция `load_document(path)` выбирает нужную по расширению файла.

In [3]:
from pypdf import PdfReader
from docx import Document as DocxDocument

def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_docx(path):
    doc = DocxDocument(path)
    parts = [p.text for p in doc.paragraphs]
    for table in doc.tables:          # таблицы хранятся отдельно от абзацев
        for row in table.rows:
            parts.append(" | ".join(cell.text for cell in row.cells))
    return "\n".join(parts)

def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_document(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return load_pdf(path)
    if ext == ".docx":
        return load_docx(path)
    if ext == ".txt":
        return load_txt(path)
    raise ValueError(f"Формат {ext} не поддерживается")

### 1.3. Очистка текста
Удаляются лишние пробелы и пустые строки, переносы строк приводятся к одному виду, склеиваются слова, разорванные дефисом, удаляются некорректные символы.

In [4]:
import re

def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\x00", "").replace("\ufffd", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

print(repr(clean_text("Текст   с  пробелами\r\n\r\n\r\nраз-\nметка")))

'Текст с пробелами\n\nразметка'


### 1.4. Разбиение на чанки
Размер чанка 500 символов, перекрытие 100 символов.

In [5]:
def split_into_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks

print([len(c) for c in split_into_chunks("а" * 1200)])

[500, 500, 400]


### 1.5. Проверка на файлах. Время и объем текста

In [6]:
import time
import pandas as pd

rows, texts = [], {}
for path in sorted(glob.glob("data/*")):
    t0 = time.perf_counter()
    raw = load_document(path)
    cleaned = clean_text(raw)
    chunks = split_into_chunks(cleaned, 500, 100)
    elapsed = time.perf_counter() - t0
    texts[os.path.basename(path)] = (raw, cleaned, chunks)
    rows.append({"файл": os.path.basename(path), "время, мс": round(elapsed * 1000, 1),
                 "символов до очистки": len(raw), "символов после": len(cleaned), "чанков": len(chunks)})

pd.DataFrame(rows)

,файл,"время, мс",символов до очистки,символов после,чанков
0,catalog.docx,8.3,136,130,1
1,guide.docx,7.4,652,651,2
2,list.pdf,8.1,172,167,1
3,note.txt,0.1,56,51,1
4,table.pdf,5.0,438,437,1
5,text.pdf,22.5,5281,5280,13


### 1.6. Результат извлечения

In [7]:
for name in ["table.pdf", "list.pdf", "guide.docx", "catalog.docx", "note.txt"]:
    raw, cleaned, chunks = texts[name]
    print("=" * 10, name, "=" * 10)
    print("до:   ", repr(raw[:260]))
    print("после:", repr(cleaned[:260]))
    print()

========== table.pdf ==========
до:    'Прайс-лист\nТовар\nЦена\nОстаток\nТовар 1\n110 руб.\n3\nТовар 2\n120 руб.\n6\nТовар 3\n130 руб.\n9\nТовар 4\n140 руб.\n12\nТовар 5\n150 руб.\n15\nТовар 6\n160 руб.\n18\nТовар 7\n170 руб.\n21\nТовар 8\n180 руб.\n24\nТовар 9\n190 руб.\n27\nТовар 10\n200 руб.\n30\nТовар 11\n210 руб.\n33\nТовар 12\n22'
после: 'Прайс-лист\nТовар\nЦена\nОстаток\nТовар 1\n110 руб.\n3\nТовар 2\n120 руб.\n6\nТовар 3\n130 руб.\n9\nТовар 4\n140 руб.\n12\nТовар 5\n150 руб.\n15\nТовар 6\n160 руб.\n18\nТовар 7\n170 руб.\n21\nТовар 8\n180 руб.\n24\nТовар 9\n190 руб.\n27\nТовар 10\n200 руб.\n30\nТовар 11\n210 руб.\n33\nТовар 12\n22'

========== list.pdf ==========
до:    'Как выбрать тетрадь:\n- смотрите на плотность бумаги\n- проверьте раз-\nметку клетки\n- выбирайте нужное количест-\nво листов\n1. Для математики: клетка\n2. Для русского: линейка\n'
после: 'Как выбрать тетрадь:\n- смотрите на плотность бумаги\n- проверьте разметку клетки\n- выбирайте нужное количество листов\n1

In [8]:
chunks = texts["text.pdf"][2]
print("чанков:", len(chunks))
print("конец 1-го :", repr(chunks[0][-100:]))
print("начало 2-го:", repr(chunks[1][:100]))
print("перекрытие совпадает:", chunks[0][-100:] == chunks[1][:100])

чанков: 13
конец 1-го : 'ание на цвет чернил и удобство корпуса. Канцелярские товары нужны\nкаждому школьнику и офисному сотру'
начало 2-го: 'ание на цвет чернил и удобство корпуса. Канцелярские товары нужны\nкаждому школьнику и офисному сотру'
перекрытие совпадает: True


### 1.7. Выводы по заданию 1

**Корректность извлечения.** Обычный текст из PDF и абзацы DOCX извлекаются полностью. Самый большой файл (около 5 тысяч символов) обрабатывается примерно за 20 мс. Слова, разорванные переносом (`раз-` и `метку`), после очистки склеиваются.

**Таблицы и списки.**
- Таблица из PDF превращается в столбик: каждая ячейка оказывается на отдельной строке (`Товар 1`, `110 руб.`, `3`), и принадлежность значений одной строке таблицы теряется.
- Таблица из DOCX читается лучше (`Ручка | 50 | синий`), но попадает в конец текста, после всех абзацев.
- Маркеры списков сохраняются как обычные символы.

**Потери форматирования.** Жирный шрифт, размеры и стили заголовков теряются, остается только текст. Сканы (PDF с картинками) не проверялись: `pypdf` не распознает текст на изображениях, для этого нужен отдельный инструмент OCR.

**Сравнение с `unstructured`.** Предыдущую версию запустить не удалось, так как ей требуется платный API-ключ, поэтому сравнение основано на описании библиотеки из семинара.

| | Собственный парсинг | unstructured |
|---|---|---|
| Скорость | Высокая, все выполняется локально | Ниже, файл передается на сервер |
| Стоимость | Бесплатно | Нужен ключ |
| Таблицы | Структура теряется | Таблицы выделяются отдельно |
| Сканы | Не поддерживаются | Поддерживаются (режим `hi_res`) |
| Заголовки и списки | Обычный текст | Определяется тип блока |

**Вывод.** Собственный парсинг лучше подходит для простых текстовых документов: он быстрый, бесплатный и полностью управляемый. Он хуже справляется с таблицами, сканами и сложной версткой. В реальном проекте его можно использовать, если документы простые, а при наличии сканов и таблиц лучше взять готовое решение или добавить OCR.

## Задание 2. Сравнение Qdrant и ChromaDB

### 2.1. Данные и векторы

Загрузить готовую модель эмбеддингов в используемой среде не удалось, поэтому векторы строятся простым способом: каждое слово по хешу попадает в одну из 384 позиций вектора, затем вектор нормируется. Документы с общими словами получают близкие векторы по косинусной мере. Обе базы получают одинаковые векторы, поэтому для сравнения этого достаточно.

Документы искусственные: 10 тем (ручки, тетради, бумага и др.), каждый документ состоит из слов одной темы, общих слов и случайного артикула. Создаются коллекции на 1000 и 8000 документов, а также 15 поисковых запросов.

In [9]:
import zlib, random, time
import numpy as np

DIM = 384

def embed(text):
    vec = np.zeros(DIM, dtype=np.float32)
    for word in text.lower().replace(",", " ").replace(".", " ").split():
        vec[zlib.crc32(word.encode("utf-8")) % DIM] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm else vec

topics = {
    "ручки": "ручка чернила стержень корпус синяя гелевая шариковая письмо пишет линия паста колпачок".split(),
    "тетради": "тетрадь клетка линейка листы обложка школа конспект запись страницы скрепка разметка поля".split(),
    "бумага": "бумага офисная формат плотность белая пачка принтер лист печать глянец матовая рулон".split(),
    "маркеры": "маркер цвет яркий выделение водная основа набор рисунок фломастер толстый тонкий скетч".split(),
    "карандаши": "карандаш графит твердость черчение набросок грифель точилка ластик мягкий рисование деревянный".split(),
    "папки": "папка регистратор документы архив механизм корешок файл хранение формат обложка кольца скоросшиватель".split(),
    "клей": "клей склеивать бумага карандаш тюбик прозрачный быстро сохнет картон поделки стик пва".split(),
    "ножницы": "ножницы лезвие резать ручки нержавеющая эргономичные острые картон бумага офисные безопасные левша".split(),
    "степлер": "степлер скобы металлический скрепление листы механизм упаковка прочный офис антистеплер булавка зажим".split(),
    "стикеры": "стикеры заметки клейкий слой напоминание яркий блок закладки листочки липкий цветные памятка".split(),
}
common = "качество цена удобно надежный новый популярный школьный офисный".split()
names = list(topics)

def make_docs(n, seed):
    rnd = random.Random(seed)
    docs = []
    for i in range(n):
        t = rnd.choice(names)
        words = rnd.sample(topics[t], 7) + rnd.sample(common, 2) + [f"арт{rnd.randint(0, 99999)}"]
        rnd.shuffle(words)
        docs.append({"id": i, "topic": t, "text": " ".join(words)})
    return docs

docs_small = make_docs(1000, seed=1)
docs_large = make_docs(8000, seed=2)

rnd = random.Random(3)
queries = [" ".join(rnd.sample(topics[names[i % 10]], 3)) for i in range(15)]
print(len(docs_small), len(docs_large), len(queries))
print(docs_small[0])
print(queries[:3])

1000 8000 15
{'id': 0, 'topic': 'бумага', 'text': 'лист популярный белая плотность принтер арт49756 глянец офисный офисная матовая'}
['корпус линия пишет', 'линейка школа скрепка', 'лист матовая глянец']


### 2.2. Функции для замеров
Потребление памяти измеряется через `psutil` (прирост памяти процесса), размер на диске считается по папке с базой.

In [10]:
import psutil, shutil

proc = psutil.Process(os.getpid())

def ram_mb():
    return proc.memory_info().rss / 1024 / 1024

def folder_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024 / 1024

shutil.rmtree("db_qdrant", ignore_errors=True)
shutil.rmtree("db_chroma", ignore_errors=True)

### 2.3. Qdrant
Используется локальный режим (данные хранятся в папке, отдельный сервер не нужен). Размер вектора 384, метрика cosine.

In [11]:
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(path="db_qdrant")

def qdrant_index(name, docs):
    qdrant.create_collection(
        collection_name=name,
        vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
    )
    ram_before = ram_mb()
    t0 = time.perf_counter()
    for start in range(0, len(docs), 500):
        batch = docs[start:start + 500]
        qdrant.upsert(name, points=[
            models.PointStruct(id=d["id"], vector=embed(d["text"]).tolist(), payload={"text": d["text"], "topic": d["topic"]})
            for d in batch])
    return time.perf_counter() - t0, ram_mb() - ram_before

q_small_time, q_small_ram = qdrant_index("collection_small", docs_small)
q_large_time, q_large_ram = qdrant_index("collection_large", docs_large)
qdrant.close()
q_disk = folder_size_mb("db_qdrant")
qdrant = QdrantClient(path="db_qdrant")
print(f"small: {q_small_time:.2f} с, large: {q_large_time:.2f} с, на диске: {q_disk:.1f} МБ")

small: 1.14 с, large: 9.34 с, на диске: 35.5 МБ


### 2.4. ChromaDB
Те же коллекции, те же векторы, метрика cosine.

In [12]:
import chromadb

chroma = chromadb.PersistentClient(path="db_chroma")

def chroma_index(name, docs):
    col = chroma.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    ram_before = ram_mb()
    t0 = time.perf_counter()
    for start in range(0, len(docs), 500):
        batch = docs[start:start + 500]
        col.add(
            ids=[str(d["id"]) for d in batch],
            embeddings=[embed(d["text"]).tolist() for d in batch],
            documents=[d["text"] for d in batch],
            metadatas=[{"topic": d["topic"]} for d in batch])
    return col, time.perf_counter() - t0, ram_mb() - ram_before

c_small, c_small_time, c_small_ram = chroma_index("collection_small", docs_small)
c_large, c_large_time, c_large_ram = chroma_index("collection_large", docs_large)
c_disk = folder_size_mb("db_chroma")
print(f"small: {c_small_time:.2f} с, large: {c_large_time:.2f} с, на диске: {c_disk:.1f} МБ")

small: 0.46 с, large: 3.69 с, на диске: 25.7 МБ


### 2.5. Поиск: 15 запросов
Каждый запрос выполняется 3 раза. Стабильность означает, что при повторе запроса результат (первые 5 документов) не меняется. Точность оценивается сравнением с полным перебором всех векторов.

In [13]:
def search_qdrant(name, q, k=5):
    res = qdrant.query_points(name, query=embed(q).tolist(), limit=k).points
    return [int(p.id) for p in res]

def search_chroma(col, q, k=5):
    res = col.query(query_embeddings=[embed(q).tolist()], n_results=k)
    return [int(i) for i in res["ids"][0]]

def run_search(func, repeats=3):
    times, stable, answers = [], 0, []
    for q in queries:
        runs = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            runs.append(func(q))
            times.append((time.perf_counter() - t0) * 1000)
        stable += all(r == runs[0] for r in runs)
        answers.append(runs[0])
    return sum(times) / len(times), stable, answers

res = {}
for size, qname, ccol in [("small", "collection_small", c_small), ("large", "collection_large", c_large)]:
    q_avg, q_stable, q_ans = run_search(lambda q: search_qdrant(qname, q))
    c_avg, c_stable, c_ans = run_search(lambda q: search_chroma(ccol, q))

    # у многих документов оценка схожести одинакова, поэтому сравниваются оценки, а не номера:
    # ответ верный, если его оценка не хуже оценки 5-го лучшего документа при полном переборе
    docs = docs_small if size == "small" else docs_large
    matrix = np.array([embed(d["text"]) for d in docs])
    def accuracy(answers):
        ok = []
        for a, q in zip(answers, queries):
            scores = matrix @ embed(q)
            best5 = np.sort(scores)[-5]
            ok.append(np.mean([scores[i] >= best5 - 1e-5 for i in a]))
        return np.mean(ok)
    q_ok, c_ok = accuracy(q_ans), accuracy(c_ans)

    res[size] = dict(q_avg=q_avg, q_stable=q_stable, c_avg=c_avg, c_stable=c_stable, q_ok=q_ok, c_ok=c_ok)
    print(f"{size}: Qdrant {q_avg:.2f} мс (стабильно {q_stable}/15, точность {q_ok:.0%}), "
          f"Chroma {c_avg:.2f} мс (стабильно {c_stable}/15, точность {c_ok:.0%})")

small: Qdrant 1.40 мс (стабильно 15/15, точность 100%), Chroma 1.22 мс (стабильно 15/15, точность 100%)


large: Qdrant 8.50 мс (стабильно 13/15, точность 100%), Chroma 1.81 мс (стабильно 15/15, точность 100%)


### 2.6. Стресс-тест
20 последовательных запросов и 5 параллельных запросов (в 5 потоках) к большой коллекции.

In [14]:
from concurrent.futures import ThreadPoolExecutor

def sequential(func, n=20):
    t0 = time.perf_counter()
    for i in range(n):
        func(queries[i % len(queries)])
    return time.perf_counter() - t0

def parallel(func, workers=5):
    errors = []
    def job(q):
        try:
            func(q)
        except Exception as e:
            errors.append(str(e))
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(job, queries[:workers]))
    return time.perf_counter() - t0, len(errors)

stress = {}
for name, f in [("Qdrant", lambda q: search_qdrant("collection_large", q)),
                ("ChromaDB", lambda q: search_chroma(c_large, q))]:
    seq = sequential(f)
    par, errs = parallel(f)
    stress[name] = (seq, par, errs)
    print(f"{name}: 20 подряд = {seq:.3f} с, 5 параллельно = {par:.3f} с, ошибок: {errs}")

Qdrant: 20 подряд = 0.149 с, 5 параллельно = 0.283 с, ошибок: 0
ChromaDB: 20 подряд = 0.032 с, 5 параллельно = 0.015 с, ошибок: 0


### 2.7. Фильтрация по метаданным
Поиск только среди документов темы «ручки».

In [15]:
q = queries[0]
r1 = qdrant.query_points("collection_large", query=embed(q).tolist(), limit=3,
        query_filter=models.Filter(must=[models.FieldCondition(key="topic", match=models.MatchValue(value="ручки"))])).points
r2 = c_large.query(query_embeddings=[embed(q).tolist()], n_results=3, where={"topic": "ручки"})
print("Qdrant  :", [(p.id, p.payload["topic"]) for p in r1])
print("ChromaDB:", list(zip(r2["ids"][0], [m["topic"] for m in r2["metadatas"][0]])))

Qdrant  : [(4334, 'ручки'), (2468, 'ручки'), (6976, 'ручки')]
ChromaDB: [('4334', 'ручки'), ('1880', 'ручки'), ('3229', 'ручки')]


### 2.8. Итоговая таблица

In [16]:
table = pd.DataFrame({
    "Qdrant": [
        f"{q_small_time:.2f}", f"{q_large_time:.2f}", f"{q_small_ram:.0f} / {q_large_ram:.0f}", f"{q_disk:.1f}",
        f"{res['small']['q_avg']:.2f}", f"{res['large']['q_avg']:.2f}", f"{res['large']['q_stable']}/15", f"{res['large']['q_ok']:.0%}",
        f"{stress['Qdrant'][0]:.3f}", f"{stress['Qdrant'][1]:.3f}"],
    "ChromaDB": [
        f"{c_small_time:.2f}", f"{c_large_time:.2f}", f"{c_small_ram:.0f} / {c_large_ram:.0f}", f"{c_disk:.1f}",
        f"{res['small']['c_avg']:.2f}", f"{res['large']['c_avg']:.2f}", f"{res['large']['c_stable']}/15", f"{res['large']['c_ok']:.0%}",
        f"{stress['ChromaDB'][0]:.3f}", f"{stress['ChromaDB'][1]:.3f}"],
}, index=[
    "Индексация small, с", "Индексация large, с", "Прирост RAM small / large, МБ", "Размер на диске (обе коллекции), МБ",
    "Поиск small, мс", "Поиск large, мс", "Стабильность (large)", "Точность топ-5 (large)",
    "20 запросов подряд, с", "5 параллельных запросов, с"])
table

,Qdrant,ChromaDB
"Индексация small, с",1.14,0.46
"Индексация large, с",9.34,3.69
"Прирост RAM small / large, МБ",14 / 27,17 / 26
"Размер на диске (обе коллекции), МБ",35.5,25.7
"Поиск small, мс",1.40,1.22
"Поиск large, мс",8.50,1.81
Стабильность (large),13/15,15/15
Точность топ-5 (large),100%,100%
"20 запросов подряд, с",0.149,0.032
"5 параллельных запросов, с",0.283,0.015


### 2.9. Выводы по заданию 2

Значения немного меняются от запуска к запуску. В проведенном эксперименте получено следующее.

- **Индексация.** ChromaDB быстрее: на 8000 документов примерно 3 с против 9 с у Qdrant.
- **Поиск.** На 1000 документов базы работают почти одинаково. На 8000 документов ChromaDB быстрее (около 1,5 мс против 8 мс). Следует учитывать, что Qdrant использовался в локальном режиме, где поиск выполняется перебором всех векторов. В режиме сервера Qdrant строит индекс, и результат может быть другим.
- **Память.** Прирост одинаковый: около 15 МБ для малой и около 26 МБ для большой коллекции.
- **Диск.** ChromaDB занимает меньше места (около 26 МБ против 36 МБ).
- **Точность и стабильность.** Обе базы вернули правильные результаты (100%). У Qdrant порядок документов иногда менялся при повторе запроса, если у нескольких документов совпадала оценка схожести.
- **Стресс-тест.** Ошибок не было ни при 20 последовательных, ни при 5 параллельных запросах.
- **Удобство API.** В ChromaDB код короче (`add` и `query`). Qdrant требует больше настроек, но дает больше контроля.
- **Фильтрация.** Обе базы поддерживают фильтры по метаданным. В Qdrant набор условий шире (`must`, `should`, диапазоны, вложенные поля), в ChromaDB фильтр задается проще, словарем `where`.

| Критерий | Лучший вариант |
|---|---|
| Небольшие проекты | ChromaDB: проще в использовании и быстрее в этом эксперименте |
| Масштабирование | Qdrant в режиме сервера: индексы, сложные фильтры, работа на нескольких машинах |

**Ограничения эксперимента.** Данные искусственные, векторы получены простым подсчетом слов, а не языковой моделью, размер базы небольшой (до 8000 документов), измерения выполнены на одном компьютере. На реальных данных и больших объемах результаты могут отличаться.